# GraviGraph - Graph analytics and graph neural networks

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "../src/GraviText/bin/Release/net10.0/Gravicode.Science.GraviText.dll"
#r "../src/GraviGraph/bin/Release/net10.0/Gravicode.Science.GraviGraph.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviGraph;
using Gravicode.Science.GraviGraph.Algorithms;
using Gravicode.Science.GraviGraph.Embeddings;
using Gravicode.Science.GraviGraph.Neural;
using Gravicode.Science.GraviNum;

var cora = Graph.Load("../datasets/cora_graph.json");
Console.WriteLine(cora);
Console.WriteLine($"features: {cora.NodeFeatures!.Shape[0]} x {cora.NodeFeatures.Shape[1]}");
Console.WriteLine($"classes : {string.Join(", ", cora.Classes)}");

## PageRank

In [ ]:
var rank = GraphAlgorithms.PageRank(cora);
Console.WriteLine($"total mass {rank.Sum():F6}");

foreach (var node in Enumerable.Range(0, cora.NodeCount).OrderByDescending(i => rank.At(i)).Take(10))
    Console.WriteLine($"node {node,6}  {rank.At(node):F6}  degree {cora.Degree(node),4}  {cora.Classes[cora.NodeLabels[node]]}");

## Structure

On a directed citation graph, connectivity means *weak* connectivity - edges followed both ways.

In [ ]:
var (weak, component) = GraphAlgorithms.ConnectedComponents(cora);
Console.WriteLine($"weakly connected  : {weak} components, largest {component.GroupBy(c => c).Max(g => g.Count())} nodes");
Console.WriteLine($"strongly connected: {GraphAlgorithms.StronglyConnectedComponents(cora).Count} components");
Console.WriteLine($"average clustering: {GraphAlgorithms.AverageClusteringCoefficient(cora):F4}");
Console.WriteLine($"modularity of the true labels: {GraphAlgorithms.Modularity(cora, cora.NodeLabels):F4}");

## Training a GCN

Only 140 of 2708 papers are labelled - the semi-supervised setting the architecture was designed for.

In [ ]:
var rng = new GraviRandom(42);
var order = rng.Permutation(cora.NodeCount);
var train = order.Take(140).ToArray();
var test = order.Skip(1708).ToArray();

var gcn = new GraphConvolutionalNetwork(hiddenSize: 16, learningRate: 0.05, epochs: 60, seed: 42)
    .Train(cora, train, features: cora.NodeFeatures);

Console.WriteLine($"train accuracy: {gcn.Score(cora, train):P1}");
Console.WriteLine($"test accuracy : {gcn.Score(cora, test):P1}");
Console.WriteLine($"majority-class baseline: {cora.NodeLabels.GroupBy(l => l).Max(g => g.Count()) / (double)cora.NodeCount:P1}");

In [ ]:
var loss = gcn.History!.Loss.ToArray();
var epochs = Enumerable.Range(0, loss.Length).Select(i => (double)i).ToArray();

var plot = new ScottPlot.Plot();
var line = plot.Add.Scatter(epochs, loss);
line.MarkerSize = 0;
plot.Title("GCN training loss");
plot.XLabel("epoch"); plot.YLabel("cross-entropy");
plot.GetPngHtml(800, 450)

## Node embeddings

Random walks need the undirected view, otherwise most walks stop after a step.

In [ ]:
var (_, comp) = GraphAlgorithms.ConnectedComponents(cora);
var biggest = comp.Select((c, i) => (c, i)).GroupBy(t => t.c).MaxBy(g => g.Count())!.Select(t => t.i).Take(400).ToArray();
var sub = cora.Subgraph(biggest).AsUndirected();

var embeddings = new Node2Vec(dimensions: 64, p: 1.0, q: 0.5, walksPerNode: 6, walkLength: 20, epochs: 3, seed: 42).Train(sub);

double same = 0, other = 0; int sameN = 0, otherN = 0;
for (var i = 0; i < sub.NodeCount; i++)
    for (var j = i + 1; j < sub.NodeCount; j++)
    {
        var s = embeddings.Similarity(i, j);
        if (sub.NodeLabels[i] == sub.NodeLabels[j]) { same += s; sameN++; } else { other += s; otherN++; }
    }
Console.WriteLine($"same-topic cosine advantage: {same / sameN - other / otherN:F4} (positive means the embedding found the topics)");

## Heterogeneous graphs

A recommendation graph has users and items; flattening them into one node set loses the thing that
made it informative. Node indices are **local to their type**, so each type can carry a different
feature width.

An edge type is the *triple*, not the relation name: `(user, rates, film)` and `(critic, rates, film)`
are different relations that happen to share a verb.


In [ ]:
var shop = new HeterogeneousGraph();
shop.AddNodeType("user", 4);
shop.AddNodeType("item", 5);

shop.AddEdge("user", "viewed", "item", 0, 0);
shop.AddEdge("user", "viewed", "item", 0, 1);
shop.AddEdge("user", "viewed", "item", 1, 1);
shop.AddEdge("user", "bought", "item", 1, 2);
shop.AddEdge("user", "bought", "item", 3, 4);

shop.SetFeatures("user", new GraviRandom(3).StandardNormal(4, 6));
shop.SetFeatures("item", new GraviRandom(5).StandardNormal(5, 3));

Console.WriteLine(shop);
Console.WriteLine($"user features {shop.Features("user").Shape[1]} wide, item features {shop.Features("item").Shape[1]} wide");

// Two numbers per edge - a score and a timestamp - which a scalar weight cannot hold.
var bought = new EdgeType("user", "bought", "item");
shop.SetEdgeFeatures(bought, NdArray.FromArray(new double[,] { { 5, 1710 }, { 3, 1840 } }));
Console.WriteLine($"edge features on '{bought}': {shop.EdgeFeatures(bought).Shape[0]} x {shop.EdgeFeatures(bought).Shape[1]}");

shop.AddReverseEdges(new EdgeType("user", "viewed", "item"));
Console.WriteLine("\nAdded the reverse as a SEPARATE relation - messages only flow along edge");
Console.WriteLine("direction, and 'user views item' deserves different weights from the converse.");

var rgcn = new RelationalConvolution(shop,
    new Dictionary<string, int> { ["user"] = 6, ["item"] = 3 }, outputSize: 8, new GraviRandom(7));

var messages = rgcn.Forward(shop, new Dictionary<string, NdArray>
{
    ["user"] = shop.Features("user"),
    ["item"] = shop.Features("item"),
});

Console.WriteLine($"\nR-GCN -> user [{string.Join(", ", messages["user"].Shape.ToArray())}], item [{string.Join(", ", messages["item"].Shape.ToArray())}]");
Console.WriteLine("In-degree is normalised PER RELATION: a user with a thousand views and three");
Console.WriteLine("purchases would otherwise lose the purchases, and the purchases are the signal.");


## Temporal graphs

A static graph says `a → b` and `b → c` imply a path from `a` to `c`. But if `b → c` happened
*before* `a → b`, nothing could have travelled that way.

Information, money and disease all obey that ordering, and a static analysis systematically
overstates what is reachable.


In [ ]:
var events = new TemporalGraph();
events.AddEdge(0, 1, time: 1);
events.AddEdge(1, 2, time: 2);
events.AddEdge(2, 3, time: 0);   // fired BEFORE anything arrived at node 2

Console.WriteLine($"statically, the edge 2->3 exists : {events.Collapse().HasEdge(2, 3)}");
Console.WriteLine($"temporally, 0 reaches            : [{string.Join(", ", events.TemporallyReachable(0).Keys.Order())}]");
Console.WriteLine("node 3 is unreachable - the 2->3 edge fired too early to carry anything\n");
Console.WriteLine($"TemporalEfficiency = {events.TemporalEfficiency():F3}   (1.0 would mean ordering never mattered)");

// Recency-weighted embedding: a static aggregation weights a year-old interaction
// exactly like yesterday's, which is why collapsed-graph recommenders get stuck.
var decayGraph = new TemporalGraph(directed: false);
decayGraph.AddEdge(0, 1, time: 0);      // old
decayGraph.AddEdge(0, 2, time: 100);    // recent

var signals = NdArray.Zeros(3, 1);
signals[1, 0] = 1.0;
signals[2, 0] = -1.0;

Console.WriteLine($"\ntime-decayed embedding of node 0 = {decayGraph.TimeDecayedFeatures(signals, asOf: 100, halfLife: 10)[0, 0]:F4}");
Console.WriteLine("negative, so the recent neighbour won");


## Graph-level pooling and classification

Node classification needs no readout; graph classification does, because graphs have different
numbers of nodes and a model needs one fixed-size vector per graph.

**The readout must not depend on node order.** Graph nodes have no canonical numbering, so a
permutation-sensitive readout would make the output depend on how the file was written.


In [ ]:
var nodeVectors = NdArray.FromArray(new double[,] { { 1, 2 }, { 3, 4 }, { 5, 6 } });
var shuffled    = NdArray.FromArray(new double[,] { { 5, 6 }, { 1, 2 }, { 3, 4 } });

foreach (var kind in new[] { PoolingKind.Mean, PoolingKind.Sum, PoolingKind.Max })
{
    var a = GraphPooling.Pool(nodeVectors, kind);
    var b = GraphPooling.Pool(shuffled, kind);
    Console.WriteLine($"{kind,-5}: [{string.Join(", ", a.ToArray())}]  order-invariant: {a.ToArray().SequenceEqual(b.ToArray())}");
}

Console.WriteLine("\nMean is size-invariant (judge composition); sum is not (size itself matters).\n");

var shapes = new List<Graph>();
var shapeLabels = new List<int>();
for (var n = 5; n <= 12; n++)
{
    shapes.Add(Graph.Cycle(n));    shapeLabels.Add(0);
    shapes.Add(Graph.Complete(n)); shapeLabels.Add(1);
}

var shapeClassifier = new GraphClassifier(inputSize: 2, hiddenSize: 16, layers: 2).Fit(shapes, shapeLabels);

Console.WriteLine($"cycles vs complete graphs: {shapeClassifier.Accuracy(shapes, shapeLabels):P1}");
Console.WriteLine($"  20-node cycle    -> class {shapeClassifier.Predict(Graph.Cycle(20))} (expected 0)");
Console.WriteLine($"  20-node complete -> class {shapeClassifier.Predict(Graph.Complete(20))} (expected 1)");
Console.WriteLine("both sizes are outside the training range, so it generalised on structure");


## Neighbourhood sampling

Full-batch message passing needs the whole graph. The problem is not memory but **neighbourhood
explosion**: a two-layer GNN on a graph of average degree 100 touches ten thousand nodes per target.

Capping the fan-out per hop makes the cost per target bounded and independent of the graph's size.


In [ ]:
var hub = new Graph(2001);
for (var i = 1; i <= 2000; i++) hub.AddEdge(0, i);

var block = NeighborSampler.Sample(hub, new[] { 0 }, new[] { 5 }, new GraviRandom(3));
Console.WriteLine($"node 0 has {hub.Degree(0)} neighbours; with fan-out 5 the batch touches {block.NodeCount} nodes\n");

var citation = Graph.Random(500, 0.02, seed: 11);
var citationFeatures = new GraviRandom(13).StandardNormal(500, 8);

var batchBlock = NeighborSampler.Sample(citation, new[] { 0, 1, 2 }, new[] { 10, 5 }, new GraviRandom(17));
var gathered = NeighborSampler.GatherFeatures(batchBlock, citationFeatures);

Console.WriteLine($"3 targets, fan-out [10, 5] on a 500-node graph:");
Console.WriteLine($"  {batchBlock.NodeCount} of 500 feature rows need to be resident");
Console.WriteLine($"  layers: {string.Join(", ", batchBlock.Layers.Select(l => l.Length + " edges"))}");

// A SAGE layer takes TWICE its feature width: self and neighbourhood are concatenated.
var sageRng = new GraviRandom(19);
var sageWeights = new[] { sageRng.StandardNormal(16, 12) * 0.1, sageRng.StandardNormal(24, 4) * 0.1 };
var sageOut = NeighborSampler.Aggregate(batchBlock, gathered, sageWeights);

Console.WriteLine($"  aggregated -> [{string.Join(", ", sageOut.Shape.ToArray())}], one row per target");


## How fast the neighbourhood explodes

Plotting the block size against fan-out makes the argument concrete: without a cap, a two-hop
neighbourhood on a moderately dense graph swallows most of it.


In [ ]:
var dense = Graph.Random(600, 0.03, seed: 23);
var fanOuts = new[] { 2, 4, 6, 8, 10, 12, 15, 20 };

var oneHop = new List<double>();
var twoHop = new List<double>();

foreach (var fan in fanOuts)
{
    oneHop.Add(NeighborSampler.Sample(dense, new[] { 0 }, new[] { fan }, new GraviRandom(29)).NodeCount);
    twoHop.Add(NeighborSampler.Sample(dense, new[] { 0 }, new[] { fan, fan }, new GraviRandom(29)).NodeCount);
}

var growth = new ScottPlot.Plot();
growth.Add.Scatter(fanOuts.Select(f => (double)f).ToArray(), oneHop.ToArray()).LegendText = "one hop";
growth.Add.Scatter(fanOuts.Select(f => (double)f).ToArray(), twoHop.ToArray()).LegendText = "two hops";

var whole = growth.Add.Scatter(new double[] { fanOuts[0], fanOuts[^1] }, new double[] { 600, 600 });
whole.LegendText = "the whole graph";
whole.LinePattern = ScottPlot.LinePattern.Dashed;
whole.MarkerSize = 0;

growth.Title("Block size against fan-out (600-node graph)");
growth.XLabel("neighbours sampled per hop");
growth.YLabel("nodes pulled into the batch");
growth.ShowLegend();
growth.GetPngHtml(750, 450)
